# STEEX LFM2.5 Fine-Tuning (Google Colab)

Fine-tune Liquid AI's LFM2.5-1.2B-Base for market prediction.
Checkpoints are saved to HF Hub so you can continue on Kaggle/Lightning.

**Free tier**: T4 GPU (16GB), ~12h sessions

In [ ]:
# Step 1: Install dependencies
!pip install -q unsloth[colab-new] datasets huggingface_hub trl

In [ ]:
# Step 2: Login to HF Hub (for checkpoint relay)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Step 3: Upload your training data
# Option A: Upload train.jsonl via Colab file picker
from google.colab import files
uploaded = files.upload()  # Upload train.jsonl
DATASET_PATH = list(uploaded.keys())[0]
print(f"Dataset: {DATASET_PATH}")

In [ ]:
# Option B: Pull dataset from HF Hub instead
# from huggingface_hub import hf_hub_download
# DATASET_PATH = hf_hub_download(repo_id="YOUR_USER/steex-training-data", filename="train.jsonl")

In [ ]:
# Step 4: Configuration
import torch

HUB_REPO = "YOUR_USERNAME/steex-lfm2-market"  # <-- Change this!
RESUME = False  # Set True to continue from last checkpoint

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Step 5: Load model with Unsloth (2x faster, 50% less memory)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="LiquidAI/LFM2.5-1.2B-Base",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Step 6: Load and format dataset
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

examples = []
with open(DATASET_PATH) as f:
    for line in f:
        examples.append(json.loads(line))

dataset = Dataset.from_list(examples)

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)
print(f"Training examples: {len(dataset)}")
print(f"Sample:\n{dataset[0]['text'][:500]}...")

In [ ]:
# Step 7: Train!
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="./checkpoints",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        save_steps=50,
        logging_steps=10,
        save_total_limit=3,
        fp16=True,
        optim="adamw_8bit",
        seed=42,
        report_to="none",
    ),
    dataset_text_field="text",
    max_seq_length=4096,
    packing=True,
)

trainer.train()

In [ ]:
# Step 8: Push checkpoint to HF Hub (relay to next platform)
model.push_to_hub(HUB_REPO, tokenizer=tokenizer, private=True)
print(f"Model pushed to https://huggingface.co/{HUB_REPO}")

In [ ]:
# Step 9: Export GGUF for local inference on Mac
model.save_pretrained_gguf(
    "steex-lfm2-market",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF exported! Download steex-lfm2-market-unsloth-Q4_K_M.gguf")

# Also push GGUF to Hub
model.push_to_hub_gguf(
    HUB_REPO + "-gguf",
    tokenizer,
    quantization_method="q4_k_m",
    private=True,
)

In [ ]:
# Step 10: Quick test
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are a quantitative trading analyst."},
    {"role": "user", "content": "VIX is at 28 (75th percentile), breadth is 40%. What regime are we in?"}
]

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
output = model.generate(inputs, max_new_tokens=256, temperature=0.3, min_p=0.15)
print(tokenizer.decode(output[0], skip_special_tokens=True))